# Preprocessing
This file:
- loads 3 corpora (`hablacultura`, `kwiziq`, and `CAES`) using `datasets` library.
- lemmatizes each text using `stanza` library.
- computes word overlap per text considering curated lexical list with CEFR annotations from Instituto Cervantes using `regex` library.
- computes readability metrics per text using `textstat` library

Input file:
- `curated_vocab.tsv`: curated lexical list from Instituto Cervantes with CEFR annotations.

Output files in `data` folder:
- `hablacultura_feats.csv`: aggregation of all metrics ready to be used in further experiments.
- `kwiziq_feats.csv`: aggregation of all metrics ready to be used in further experiments.
- `caes_feats.csv`: aggregation of all metrics ready to be used in further experiments.

In [21]:
import os
import re
import pandas as pd
import numpy as np

import stanza

import datasets
from datasets import load_dataset

import matplotlib.pyplot as plt
import seaborn as sns

import textstat

from tqdm import tqdm

## Corpus

In [23]:
datasets = ["lmvasque/hablacultura", "lmvasque/kwiziq", "UniversalCEFR/caes_es"]
dts = []
for ds in datasets:
    dts.append((ds, load_dataset(ds)))

In [24]:
dts

[('lmvasque/hablacultura',
  DatasetDict({
      train: Dataset({
          features: ['category', 'text', 'level', 'level-3'],
          num_rows: 713
      })
  })),
 ('lmvasque/kwiziq',
  DatasetDict({
      train: Dataset({
          features: ['category', 'text', 'level', 'level-3'],
          num_rows: 206
      })
  })),
 ('UniversalCEFR/caes_es',
  DatasetDict({
      train: Dataset({
          features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text'],
          num_rows: 31149
      })
  }))]

In [4]:
# iterate and print two samples from each
for name, ds_dict in dts:
    print(f"\n--- Dataset {name} ---")
    train_ds = ds_dict["train"]
    samples = train_ds.shuffle(seed=42).select(range(2))
    for j, sample in enumerate(samples):
        print(f"\nSample {j + 1}:")
        for key, value in sample.items():
            print(f"{key}: {value}")


--- Dataset lmvasque/hablacultura ---

Sample 1:
category: B1
text: Fundado en 1858 por el francés Jean Touan, el primer emplazamiento del Café fue en la esquina de las calles Rivadavia y Esmeralda, denominándose Tortoni en referencia a un local de París, en el Boulevard des Italiens, donde se reunía la élite de la cultura parisina en el siglo XIX.
level: simple
level-3: intermediate

Sample 2:
category: B2
text: En el colegio, el psicólogo les dijo que no pensaba tomar medidas y no hicieron caso a los padres de Eva.
level: complex
level-3: intermediate

--- Dataset lmvasque/kwiziq ---

Sample 1:
category: A1
text: Rafa Nadal es una estrella del tenis español
que tiene fama internacional.
Es de Manacor, en las islas Baleares.
Tiene 35 años, mide 1.85
y pesa 85 kilos.
Tiene el pelo de color castaño,
liso y un poco largo.
Es muy simpático y divertido,
pero también es tímido,
humilde y respetuoso.
Su mujer se llama Mery
y es paciente y discreta
con la vida de Rafa.
En su tiempo libre
a R

## Data Cleaning

In [5]:
from datasets import Dataset

def clean_ds(ds, columns):
    before = len(ds)
    
    df = ds.to_pandas()
    
    # remove NaN / None
    df = df.dropna(subset=columns)
    df = df.dropna(subset=["text"])
    
    # remove duplicates
    df = df.drop_duplicates(subset=columns)
    df = df.drop_duplicates(subset=["text"])
    
    after = len(df)
    print(f"Removed {before - after} rows ({before} → {after})")
    
    return Dataset.from_pandas(df, preserve_index=False)

for name, ds_dict in dts:
    cols = {
        "lmvasque/hablacultura": ['category', 'text'],
        "lmvasque/kwiziq": ['category', 'text'],
        "UniversalCEFR/caes_es": ['cefr_level', 'text'],
    }
    ds_dict["train"] = clean_ds(ds_dict["train"], cols[name])
    

Removed 45 rows (713 → 668)
Removed 0 rows (206 → 206)
Removed 10518 rows (31149 → 20631)


## Lemmatizing

In [6]:
nlp = stanza.Pipeline(lang="es", processors="tokenize,lemma", use_gpu=True)

2026-03-30 19:05:08 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-03-30 19:05:08 INFO: Downloaded file to /home/dimm/stanza_resources/resources.json
2026-03-30 19:05:08 WARNING: Language es package default expects mwt, which has been added
2026-03-30 19:05:08 INFO: Loading these models for language: es (Spanish):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| lemma     | combined_nocharlm |

2026-03-30 19:05:08 INFO: Using device: cuda
2026-03-30 19:05:08 INFO: Loading: tokenize
2026-03-30 19:05:09 INFO: Loading: mwt
2026-03-30 19:05:09 INFO: Loading: lemma
2026-03-30 19:05:09 INFO: Done loading processors!


In [7]:
outputcsv = {
    "lmvasque/hablacultura": "hablacultura",
    "lmvasque/kwiziq": "kwiziq",
    "UniversalCEFR/caes_es": "caes",
}

for name, ds_dict in dts:
    corpus = [] 
    
    data = ds_dict["train"]
    
    for sample in tqdm(data, desc=name):
        text = sample["text"]
        text = text.replace("_", " ")
        
        doc = nlp(text)
        lemmas = " ".join(
            [word.lemma for sent in doc.sentences for word in sent.words]
        )
        
        if "caes" in name:
            cefr = sample["cefr_level"]
        else:
            cefr = sample["category"]
        
        corpus.append([text, lemmas, cefr])
    
    df = pd.DataFrame(corpus, columns=["text", "lemmas", "cefr"])
    
    print(f"\n{name}")
    print("nan", df.isna().sum())
    
    df.to_csv(os.path.join("data", f"{outputcsv[name]}.csv"), index=False)

lmvasque/hablacultura: 100%|██████████████████| 668/668 [00:11<00:00, 59.69it/s]



lmvasque/hablacultura
nan text      0
lemmas    0
cefr      0
dtype: int64


lmvasque/kwiziq: 100%|████████████████████████| 206/206 [00:08<00:00, 24.26it/s]



lmvasque/kwiziq
nan text      0
lemmas    0
cefr      0
dtype: int64


UniversalCEFR/caes_es: 100%|██████████████| 20631/20631 [07:32<00:00, 45.54it/s]



UniversalCEFR/caes_es
nan text      0
lemmas    0
cefr      0
dtype: int64


In [8]:
files = ["hablacultura", "kwiziq", "caes"]

for file in files:
    df = pd.read_csv(os.path.join("data", file+".csv"))
    df["cefr"] = df["cefr"].where(df["cefr"].isin(["A1","A2","B1","B2", "C1", "C2"]), np.nan)
    print(df["cefr"].value_counts().sort_index())

cefr
A2     33
B1    245
B2    273
C1     99
Name: count, dtype: int64
cefr
A1    21
A2    33
B1    54
B2    59
C1    39
Name: count, dtype: int64
cefr
A1    5553
A2    5709
B1    4483
B2    3100
C1    1786
Name: count, dtype: int64


## Co-occurrence
Overlap with list of words from CEFR lists

In [5]:
import pandas as pd
import numpy as np

lista2 = pd.read_csv('curated_vocab.tsv', sep='\t')
lista2.columns = ['lema', 'desambiguacion_combinatoria', 'desambiguacion_contexto_polisemia', 'nivel', 'ontologia_num', 'ontologia_cat', 'ontologia_subnum', 'ontologia_subcat']
lista2

,lema,desambiguacion_combinatoria,desambiguacion_contexto_polisemia,nivel,ontologia_num,ontologia_cat,ontologia_subnum,ontologia_subcat
0,pelo,NaN,NaN,A1,1,Individuo: dimensión física,1.1.,Partes del cuerpo
1,ojo,NaN,NaN,A1,1,Individuo: dimensión física,1.1.,Partes del cuerpo
2,nariz,NaN,NaN,A1,1,Individuo: dimensión física,1.1.,Partes del cuerpo
3,barba,NaN,NaN,A1,1,Individuo: dimensión física,1.2.,Características físicas
4,bigote,NaN,NaN,A1,1,Individuo: dimensión física,1.2.,Características físicas
...,...,...,...,...,...,...,...,...
4180,alimento transgénico,NaN,NaN,B2,20,Geografía y naturaleza,20.7.,Problemas medioambientales y desastres naturales
4181,agricultura ecológica,NaN,NaN,B2,20,Geografía y naturaleza,20.7.,Problemas medioambientales y desastres naturales
4182,radiación solar,exponerse a/protegerse de,NaN,B2,20,Geografía y naturaleza,20.7.,Problemas medioambientales y desastres naturales
4183,recurso natural,aprovechar/agotarse,NaN,B2,20,Geografía y naturaleza,20.7.,Problemas medioambientales y desastres naturales


In [6]:
lista2["lema"] = lista2["lema"].str.strip().replace("\s+", "", regex=True)

<>:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
/tmp/ipykernel_18060/3602345476.py:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  lista2["lema"] = lista2["lema"].str.strip().replace("\s+", "", regex=True)


In [7]:
lista2["nivel"].value_counts().sort_index()

nivel
A1     485
A2     575
A3       1
A4       1
B1    1182
B2    1940
Name: count, dtype: int64

In [8]:
lista2["nivel"] = lista2["nivel"].where(lista2["nivel"].isin(["A1","A2","B1","B2"]), np.nan)
lista2["nivel"].value_counts().sort_index()

nivel
A1     485
A2     575
B1    1182
B2    1940
Name: count, dtype: int64

In [13]:
lista2.dropna(subset=["lema", "nivel"], inplace=True)
lista2.shape

(4175, 8)

In [14]:
lista2[["desambiguacion_combinatoria", "desambiguacion_contexto_polisemia"]].isna().sum()

desambiguacion_combinatoria          2507
desambiguacion_contexto_polisemia    4046
dtype: int64

In [15]:
lista2["lema"].str.count(" ").value_counts()

lema
0    4175
Name: count, dtype: int64

In [10]:
lista2.groupby("nivel").sample(n=3, random_state=42)[["lema", "nivel"]]

,lema,nivel
419,monumento,A1
475,invierno,A1
181,herramienta,A1
511,sed,A2
702,internet,A2
826,gramo,A2
1119,conservador,B1
1833,carril,B1
1615,electrodoméstico,B1
3984,hinduismo,B2


## Scoring Co-occurrence

In [16]:
def score_cooc(df):
    
    def count_overlap(text: str, pattern: re.Pattern) -> int:
        # return the number of matches of the compiled regex in text.
        return len(pattern.findall(text))
    
    def compile_level_pattern(lemmas_series: pd.Series) -> re.Pattern:
        # compile a single regex for all lemmas in a level (case-insensitive, whole words).
        lemmas_clean = (
            lemmas_series.dropna()
            .astype(str)
            .str.strip()
            .unique()
            .tolist()
        )
        if not lemmas_clean:
            # compiles a regex that matches nothing
            return re.compile(r"(?!x)x")
            
        # Sort by length to slightly reduce backtracking on large alternations
        lemmas_sorted = sorted(lemmas_clean, key=len, reverse=True)
        return re.compile(r"\b(" + "|".join(map(re.escape, lemmas_sorted)) + r")\b", flags=re.IGNORECASE)
    
    # build lemma set per level
    levels_order = ["A1", "A2", "B1", "B2"]
    lemmas = {
        nivel: lista2.loc[lista2["nivel"].astype(str) == nivel, "lema"]
        for nivel in levels_order
    }
    
    print("Count of lemmas per level from curated vocab list:")
    for nivel in levels_order:
        print(nivel, lemmas[nivel].dropna().astype(str).str.strip().nunique())
    
    # compile single regex per level
    patterns = {nivel: compile_level_pattern(lemmas[nivel]) for nivel in levels_order}
    
    # count per text
    texts = df["lemmas"].fillna("").astype(str)
    rows = []
    
    for text in tqdm(texts, desc="Counting lemmas per level", unit="text"):
        row_counts = {}
        for nivel in levels_order:
            row_counts[nivel] = count_overlap(text, patterns[nivel])
        rows.append(row_counts)
    
    # result: df with texts on rows and levels on columns
    level_counts = pd.DataFrame(rows, index=df.index)[levels_order].fillna(0).astype(int)
    
    dfc = pd.concat([df, level_counts], axis=1).rename(
        columns={
            "A1": "A1_overlap",
            "A2": "A2_overlap",
            "B1": "B1_overlap",
            "B2": "B2_overlap"
        }
    )
    
    for cefr, dfg in dfc.groupby("cefr"):
        print(f"\nCEFR: {cefr}")
        print(
            print(dfg[["A1_overlap", "A2_overlap", "B1_overlap", "B2_overlap"]].sum())
        )
    print()
    
    return dfc

dfcs = {}
for file in files:
    print(file.upper())
    df = pd.read_csv(os.path.join("data", file+".csv"))
    df = df.dropna(subset=["text"])
    df["cefr"] = df["cefr"].where(df["cefr"].isin(["A1","A2","B1","B2", "C1", "C2"]), np.nan)
    dfcs[file] = score_cooc(df)
    

HABLACULTURA
Count of lemmas per level from curated vocab list:
A1 305
A2 462
B1 944
B2 1512


Counting lemmas per level: 100%|███████████| 668/668 [00:01<00:00, 425.30text/s]



CEFR: A2
A1_overlap    106
A2_overlap     92
B1_overlap    113
B2_overlap    156
dtype: int64
None

CEFR: B1
A1_overlap    1092
A2_overlap     785
B1_overlap    1124
B2_overlap    1658
dtype: int64
None

CEFR: B2
A1_overlap    1118
A2_overlap     791
B1_overlap    1262
B2_overlap    1861
dtype: int64
None

CEFR: C1
A1_overlap    427
A2_overlap    254
B1_overlap    464
B2_overlap    666
dtype: int64
None

KWIZIQ
Count of lemmas per level from curated vocab list:
A1 305
A2 462
B1 944
B2 1512


Counting lemmas per level: 100%|███████████| 206/206 [00:01<00:00, 127.93text/s]



CEFR: A1
A1_overlap    401
A2_overlap    260
B1_overlap    312
B2_overlap    454
dtype: int64
None

CEFR: A2
A1_overlap    502
A2_overlap    338
B1_overlap    464
B2_overlap    734
dtype: int64
None

CEFR: B1
A1_overlap     829
A2_overlap     601
B1_overlap     923
B2_overlap    1354
dtype: int64
None

CEFR: B2
A1_overlap     844
A2_overlap     649
B1_overlap     945
B2_overlap    1500
dtype: int64
None

CEFR: C1
A1_overlap     789
A2_overlap     509
B1_overlap     764
B2_overlap    1212
dtype: int64
None

CAES
Count of lemmas per level from curated vocab list:
A1 305
A2 462
B1 944
B2 1512


Counting lemmas per level: 100%|███████| 20630/20630 [01:32<00:00, 222.28text/s]


CEFR: A1
A1_overlap    56798
A2_overlap    24088
B1_overlap    34814
B2_overlap    65157
dtype: int64
None

CEFR: A2
A1_overlap    66589
A2_overlap    35300
B1_overlap    47976
B2_overlap    78151
dtype: int64
None

CEFR: B1
A1_overlap    49758
A2_overlap    31795
B1_overlap    47801
B2_overlap    86460
dtype: int64
None

CEFR: B2
A1_overlap    46184
A2_overlap    24476
B1_overlap    47444
B2_overlap    84585
dtype: int64
None

CEFR: C1
A1_overlap    28304
A2_overlap    16004
B1_overlap    32786
B2_overlap    47275
dtype: int64
None



## Textstat

In [17]:
textstat.set_lang("es") # setting spanish language

In [18]:
def calculate_textstat(df):
    
    # Spanish metrics and other length-based metrics
    def calculate_spanish_metrics(text):
        """Calculate various Spanish readability metrics using textstat"""
        metrics = {
            'fernandez_huerta': textstat.fernandez_huerta(text),
            'szigriszt_pazos': textstat.szigriszt_pazos(text),
            'gutierrez_polini': textstat.gutierrez_polini(text),
            'crawford': textstat.crawford(text),
            'syllable_count': textstat.syllable_count(text),
            'lexicon_count': textstat.lexicon_count(text), # basically number of words
            'sentence_count': textstat.sentence_count(text),
            'words_per_sentence': textstat.words_per_sentence(text),
            'avg_syllables_per_word': textstat.avg_syllables_per_word(text),
            'difficult_words': textstat.difficult_words(text) # word with more than 2 syllables and not in a easy-word list
        }
        return metrics

    metrics_list = []
    for idx, row in df.iterrows():
        metrics = calculate_spanish_metrics(row['text'])
        metrics_list.append(metrics)
    
    # convert to DataFrame and concatenate with original df
    metrics_df = pd.DataFrame(metrics_list, index=df.index)
    df_with_metrics = pd.concat([df, metrics_df], axis=1)

    # summary table of metrics by level
    summary_metrics = df_with_metrics.groupby('cefr').agg({
        'fernandez_huerta': ['mean', 'std'],
        'szigriszt_pazos': ['mean', 'std'],
        'gutierrez_polini': ['mean', 'std'],
        'crawford': ['mean', 'std'],
        'words_per_sentence': ['mean', 'std'],
        'avg_syllables_per_word': ['mean', 'std'],
        'difficult_words': ['mean', 'std'],
        'lexicon_count': ['mean', 'std']
    }).round(2)
    
    summary_metrics.columns = ['_'.join(col).strip() for col in summary_metrics.columns.values]
    summary_metrics = summary_metrics.reset_index()
    summary_metrics['cefr'] = summary_metrics['cefr'].str.upper()
    print(summary_metrics)

    return df_with_metrics

dfts = {}
for file in files:
    print(file.upper())
    df = pd.read_csv(os.path.join("data", file+".csv"))
    df = df.dropna(subset=["text"])
    df["cefr"] = df["cefr"].where(df["cefr"].isin(["A1","A2","B1","B2", "C1", "C2"]), np.nan)
    dfts[file] = calculate_textstat(df)

HABLACULTURA
  cefr  fernandez_huerta_mean  fernandez_huerta_std  szigriszt_pazos_mean  \
0   A2                  78.96                 11.76                 75.05   
1   B1                  71.47                 13.98                 67.62   
2   B2                  67.96                 16.14                 64.09   
3   C1                  67.81                 19.79                 64.01   

   szigriszt_pazos_std  gutierrez_polini_mean  gutierrez_polini_std  \
0                11.87                  44.68                  4.48   
1                14.05                  42.12                  5.33   
2                16.26                  40.49                  6.17   
3                19.85                  40.56                  7.51   

   crawford_mean  crawford_std  words_per_sentence_mean  \
0           4.23          0.81                    16.91   
1           4.68          0.91                    22.69   
2           4.90          0.97                    24.53   
3        

In [19]:
for file in files:
    print(file.upper())
    
    df = pd.read_csv(os.path.join("data", file + ".csv"))
    df = df.dropna(subset=["text"])
    
    dfc = dfcs[file]
    df_with_metrics = dfts[file]
        
    data = pd.merge(
        df,
        dfc,
        on=["text", "lemmas", "cefr"],
        how="inner"
    )
    
    data = pd.merge(
        data,
        df_with_metrics,
        on=["text", "lemmas", "cefr"],
        how="inner"
    )

    data.to_csv(os.path.join("data", file+"_feats.csv"), index=False)

HABLACULTURA
KWIZIQ
CAES


In [20]:
outputfiles = ["caes_feats", "hablacultura_feats", "kwiziq_feats"]
for file in outputfiles:
    df = pd.read_csv(os.path.join("data", file + ".csv"))
    
    # treat empty strings as NaN
    df = df.replace("", pd.NA)
    
    print(f"\n--- {file} ---")
    print(df.isna().sum())


--- caes_feats ---
text                      0
lemmas                    0
cefr                      0
A1_overlap                0
A2_overlap                0
B1_overlap                0
B2_overlap                0
fernandez_huerta          0
szigriszt_pazos           0
gutierrez_polini          0
crawford                  0
syllable_count            0
lexicon_count             0
sentence_count            0
words_per_sentence        0
avg_syllables_per_word    0
difficult_words           0
dtype: int64

--- hablacultura_feats ---
text                      0
lemmas                    0
cefr                      0
A1_overlap                0
A2_overlap                0
B1_overlap                0
B2_overlap                0
fernandez_huerta          0
szigriszt_pazos           0
gutierrez_polini          0
crawford                  0
syllable_count            0
lexicon_count             0
sentence_count            0
words_per_sentence        0
avg_syllables_per_word    0
difficult_words